# Day 2: Statistics, Probability & Visualization
## Stage 1: Fundamental Data Science & Analytics — Epson Training

**Learning Objectives:**
- Apply descriptive statistics to production data
- Understand and use normal and binomial probability distributions
- Create insightful visualizations with Matplotlib and Seaborn
- Analyze Epson production trends and machine quality

> 💡 **Google Colab Tip:** Run the first code cell to install/verify dependencies.

In [ ]:
# Install / verify libraries (already available in Colab)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from datetime import datetime, timedelta

# Consistent style
sns.set_theme(style="whitegrid", palette="muted")
np.random.seed(42)

print("✅ All libraries loaded successfully")

---
## Part 1: Generate Epson Production Dataset

In [ ]:
# Generate 30 days of hourly production data
n_hours = 24 * 30   # 720 records
start = datetime(2024, 1, 1, 0, 0)
timestamps = [start + timedelta(hours=i) for i in range(n_hours)]

machines = ["Line_A", "Line_B", "Line_C", "Line_D"]

df = pd.DataFrame({
    "timestamp": timestamps,
    "machine_id": np.tile(machines, n_hours // len(machines) + 1)[:n_hours],
    "temperature_c": np.round(np.random.normal(72, 3, n_hours), 2),
    "vibration_mms": np.round(np.abs(np.random.normal(3, 1.2, n_hours)), 3),
    "production_units": np.random.randint(180, 220, n_hours),
    "defect_count": np.random.randint(0, 8, n_hours),
    "downtime_min": np.where(np.random.rand(n_hours) < 0.1,
                             np.random.randint(5, 60, n_hours), 0),
})

df["defect_rate"] = np.round(df["defect_count"] / df["production_units"] * 100, 2)
df["hour"] = df["timestamp"].dt.hour
df["day_of_week"] = df["timestamp"].dt.day_name()
df["week"] = df["timestamp"].dt.isocalendar().week.astype(int)
df["shift"] = df["hour"].apply(lambda h: "Day" if 6 <= h < 18 else "Night")

print(f"Dataset shape: {df.shape}")
df.head()

---
## Part 2: Descriptive Statistics

Descriptive statistics summarize and describe the main features of a dataset.

In [ ]:
# --- Central Tendency & Dispersion ---
numeric_cols = ["temperature_c", "vibration_mms", "production_units", "defect_rate"]

print("=" * 60)
print("  DESCRIPTIVE STATISTICS — Epson Production Data")
print("=" * 60)

for col in numeric_cols:
    data = df[col]
    print(f"\n📊 {col}")
    print(f"  Mean:     {data.mean():.3f}")
    print(f"  Median:   {data.median():.3f}")
    print(f"  Mode:     {data.mode().iloc[0]:.3f}")
    print(f"  Std Dev:  {data.std():.3f}")
    print(f"  Variance: {data.var():.3f}")
    print(f"  Range:    [{data.min():.3f}, {data.max():.3f}]")
    print(f"  IQR:      {data.quantile(0.75) - data.quantile(0.25):.3f}")
    print(f"  Skewness: {data.skew():.3f}")
    print(f"  Kurtosis: {data.kurtosis():.3f}")

In [ ]:
# --- Percentiles and Quartiles ---
print("=== Temperature Percentiles ===")
percentiles = [10, 25, 50, 75, 90, 95, 99]
for p in percentiles:
    val = np.percentile(df["temperature_c"], p)
    print(f"  P{p:2d}: {val:.2f}°C")

print("\n=== Machine-Level Summary ===")
print(df.groupby("machine_id")[numeric_cols].agg(["mean", "std"]).round(2))

---
## Part 3: Probability Distributions

### 3.1 Normal Distribution

In [ ]:
from scipy.stats import norm, binom

# Normal distribution — Temperature model
mu = df["temperature_c"].mean()
sigma = df["temperature_c"].std()

print(f"Temperature: μ = {mu:.2f}°C, σ = {sigma:.2f}°C")

# Probability calculations
prob_above_78 = 1 - norm.cdf(78, loc=mu, scale=sigma)
prob_below_65 = norm.cdf(65, loc=mu, scale=sigma)
prob_normal_range = norm.cdf(78, loc=mu, scale=sigma) - norm.cdf(65, loc=mu, scale=sigma)

print(f"\nP(Temperature > 78°C):          {prob_above_78:.4f} = {prob_above_78*100:.2f}%")
print(f"P(Temperature < 65°C):          {prob_below_65:.4f} = {prob_below_65*100:.2f}%")
print(f"P(65°C ≤ Temperature ≤ 78°C):   {prob_normal_range:.4f} = {prob_normal_range*100:.2f}%")

# 68-95-99.7 Rule
print("\n=== Empirical Rule (68-95-99.7) ===")
for n_sigma in [1, 2, 3]:
    lo = mu - n_sigma * sigma
    hi = mu + n_sigma * sigma
    p = norm.cdf(hi, mu, sigma) - norm.cdf(lo, mu, sigma)
    actual = ((df["temperature_c"] >= lo) & (df["temperature_c"] <= hi)).mean()
    print(f"  μ ± {n_sigma}σ = [{lo:.1f}, {hi:.1f}]: theoretical {p*100:.1f}%, actual {actual*100:.1f}%")

In [ ]:
# Visualize Normal Distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram with fitted normal curve
x_range = np.linspace(df["temperature_c"].min() - 2, df["temperature_c"].max() + 2, 200)
axes[0].hist(df["temperature_c"], bins=30, density=True, alpha=0.6,
             color="steelblue", edgecolor="white", label="Observed data")
axes[0].plot(x_range, norm.pdf(x_range, mu, sigma), "r-", lw=2.5, label=f"Normal(μ={mu:.1f}, σ={sigma:.1f})")
axes[0].axvline(78, color="orange", linestyle="--", label="Warning (78°C)")
axes[0].set_title("Temperature Distribution with Normal Fit")
axes[0].set_xlabel("Temperature (°C)")
axes[0].set_ylabel("Density")
axes[0].legend()

# Q-Q Plot to check normality
stats.probplot(df["temperature_c"], dist="norm", plot=axes[1])
axes[1].set_title("Q-Q Plot — Temperature (Normality Check)")
axes[1].get_lines()[0].set(markersize=2, alpha=0.5)

plt.tight_layout()
plt.savefig("day2_normal_distribution.png", dpi=100, bbox_inches="tight")
plt.show()

### 3.2 Binomial Distribution

In [ ]:
# Binomial distribution — Quality inspection model
# Each unit has a probability p of being defective
p_defect = df["defect_rate"].mean() / 100    # convert % to probability
n_units = 200                                  # units inspected per batch

print(f"Quality Inspection Model:")
print(f"  Units per batch (n):   {n_units}")
print(f"  Defect probability (p): {p_defect:.4f} ({p_defect*100:.2f}%)")
print(f"  Expected defects:       {n_units * p_defect:.1f}")
print(f"  Std Dev:                {np.sqrt(n_units * p_defect * (1 - p_defect)):.2f}")

# Probability calculations
prob_0_defects = binom.pmf(0, n_units, p_defect)
prob_le5 = binom.cdf(5, n_units, p_defect)
prob_gt10 = 1 - binom.cdf(10, n_units, p_defect)

print(f"\nP(exactly 0 defects in batch): {prob_0_defects:.6f}")
print(f"P(≤ 5 defects in batch):       {prob_le5:.4f} = {prob_le5*100:.2f}%")
print(f"P(> 10 defects in batch):      {prob_gt10:.4f} = {prob_gt10*100:.2f}%")

In [ ]:
# Visualize Binomial Distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# PMF
k_values = np.arange(0, 20)
pmf_values = binom.pmf(k_values, n_units, p_defect)
axes[0].bar(k_values, pmf_values, color="#4C72B0", edgecolor="white", alpha=0.8)
axes[0].axvline(n_units * p_defect, color="red", linestyle="--", label=f"Mean = {n_units*p_defect:.1f}")
axes[0].set_title(f"Binomial PMF: Defects per Batch\n(n={n_units}, p={p_defect:.4f})")
axes[0].set_xlabel("Number of Defects")
axes[0].set_ylabel("Probability")
axes[0].legend()

# CDF
cdf_values = binom.cdf(k_values, n_units, p_defect)
axes[1].step(k_values, cdf_values, where="post", color="#C44E52", linewidth=2)
axes[1].axhline(0.95, color="gray", linestyle=":", label="95th percentile")
axes[1].set_title("Binomial CDF: Defects per Batch")
axes[1].set_xlabel("Number of Defects")
axes[1].set_ylabel("Cumulative Probability")
axes[1].legend()

plt.tight_layout()
plt.savefig("day2_binomial_distribution.png", dpi=100, bbox_inches="tight")
plt.show()

---
## Part 4: Visualization with Matplotlib and Seaborn

### 4.1 Matplotlib — Fine-Grained Control

In [ ]:
# Production trend over time (weekly aggregation)
df_daily = df.groupby(df["timestamp"].dt.date).agg(
    total_production=("production_units", "sum"),
    avg_temperature=("temperature_c", "mean"),
    total_defects=("defect_count", "sum")
).reset_index()
df_daily["date"] = pd.to_datetime(df_daily["timestamp"])
df_daily["defect_rate_daily"] = df_daily["total_defects"] / df_daily["total_production"] * 100

fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)
fig.suptitle("Epson Production Trends — January 2024", fontsize=14, fontweight="bold")

# Production volume
axes[0].fill_between(df_daily["date"], df_daily["total_production"],
                     alpha=0.4, color="steelblue")
axes[0].plot(df_daily["date"], df_daily["total_production"], color="steelblue", linewidth=1.5)
axes[0].set_ylabel("Daily Production (units)")
axes[0].set_title("Daily Production Volume")

# Average temperature
axes[1].plot(df_daily["date"], df_daily["avg_temperature"], color="tomato", linewidth=1.5)
axes[1].axhline(72, color="gray", linestyle="--", linewidth=0.8, label="Target (72°C)")
axes[1].fill_between(df_daily["date"], 70, 74, alpha=0.1, color="green", label="Normal band")
axes[1].set_ylabel("Avg Temperature (°C)")
axes[1].set_title("Average Daily Temperature")
axes[1].legend()

# Defect rate
colors = ["red" if r > 2 else "green" for r in df_daily["defect_rate_daily"]]
axes[2].bar(df_daily["date"], df_daily["defect_rate_daily"], color=colors, width=0.8)
axes[2].axhline(2, color="orange", linestyle="--", linewidth=1, label="Threshold (2%)")
axes[2].set_ylabel("Defect Rate (%)")
axes[2].set_xlabel("Date")
axes[2].set_title("Daily Defect Rate")
axes[2].legend()
axes[2].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.savefig("day2_production_trends.png", dpi=100, bbox_inches="tight")
plt.show()

### 4.2 Seaborn — Statistical Visualizations

In [ ]:
# Seaborn: Distribution plots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Seaborn Statistical Visualizations — Machine Quality", fontsize=13, fontweight="bold")

# 1. KDE + Histogram per machine
for machine in machines:
    subset = df[df["machine_id"] == machine]["temperature_c"]
    axes[0, 0].hist(subset, bins=20, alpha=0.4, label=machine, density=True)
axes[0, 0].set_title("Temperature Distribution by Machine")
axes[0, 0].set_xlabel("Temperature (°C)")
axes[0, 0].legend()

# 2. Box plot: defect rate by machine
sns.boxplot(data=df, x="machine_id", y="defect_rate", ax=axes[0, 1], palette="Set2")
axes[0, 1].set_title("Defect Rate Distribution by Machine")
axes[0, 1].set_xlabel("Machine")
axes[0, 1].set_ylabel("Defect Rate (%)")

# 3. Violin plot: temperature by shift
sns.violinplot(data=df, x="shift", y="temperature_c", ax=axes[1, 0],
               palette="pastel", inner="quartile")
axes[1, 0].set_title("Temperature by Shift (Day vs Night)")
axes[1, 0].set_ylabel("Temperature (°C)")

# 4. Scatter: temperature vs defect rate with regression
sns.regplot(data=df.sample(200), x="temperature_c", y="defect_rate",
            ax=axes[1, 1], scatter_kws={"alpha": 0.4, "s": 20},
            line_kws={"color": "red"})
axes[1, 1].set_title("Temperature vs Defect Rate")
axes[1, 1].set_xlabel("Temperature (°C)")
axes[1, 1].set_ylabel("Defect Rate (%)")

plt.tight_layout()
plt.savefig("day2_seaborn_plots.png", dpi=100, bbox_inches="tight")
plt.show()

In [ ]:
# Seaborn: Correlation heatmap
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Heatmap
corr_matrix = df[["temperature_c", "vibration_mms", "production_units",
                   "defect_count", "defect_rate", "downtime_min"]].corr()
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm",
            center=0, square=True, ax=axes[0], cbar_kws={"shrink": 0.8})
axes[0].set_title("Correlation Heatmap — Production Variables")

# Bar chart: production by day of week
day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
dow_data = df.groupby("day_of_week")["production_units"].mean().reindex(day_order)
sns.barplot(x=dow_data.index, y=dow_data.values, ax=axes[1], palette="Blues_d")
axes[1].set_title("Average Production by Day of Week")
axes[1].set_xlabel("Day")
axes[1].set_ylabel("Avg Production (units)")
axes[1].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.savefig("day2_heatmap_dow.png", dpi=100, bbox_inches="tight")
plt.show()

---
## Part 5: Case Study — Epson Production Trends and Machine Quality

In [ ]:
# Hypothesis test: Is Line_A's defect rate significantly different from Line_B's?
line_a_defects = df[df["machine_id"] == "Line_A"]["defect_rate"]
line_b_defects = df[df["machine_id"] == "Line_B"]["defect_rate"]

t_stat, p_value = stats.ttest_ind(line_a_defects, line_b_defects)

print("=" * 55)
print("  Hypothesis Test: Line_A vs Line_B Defect Rate")
print("=" * 55)
print(f"  H₀: Mean defect rates are equal")
print(f"  H₁: Mean defect rates are different")
print(f"\n  Line_A mean defect rate: {line_a_defects.mean():.3f}%")
print(f"  Line_B mean defect rate: {line_b_defects.mean():.3f}%")
print(f"\n  t-statistic: {t_stat:.4f}")
print(f"  p-value:     {p_value:.4f}")

alpha = 0.05
if p_value < alpha:
    print(f"\n  ✅ Reject H₀ (p < {alpha}): Significant difference detected")
else:
    print(f"\n  ⚠️  Fail to reject H₀ (p ≥ {alpha}): No significant difference")

In [ ]:
# Final summary dashboard
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("Epson Production Quality Dashboard — January 2024",
             fontsize=14, fontweight="bold")

# 1. Weekly production trend
weekly = df.groupby("week")["production_units"].sum()
axes[0, 0].plot(weekly.index, weekly.values, marker="o", color="steelblue")
axes[0, 0].set_title("Weekly Production Volume")
axes[0, 0].set_xlabel("Week")
axes[0, 0].set_ylabel("Units")

# 2. Defect rate by machine
mean_defect = df.groupby("machine_id")["defect_rate"].mean()
bars = axes[0, 1].bar(mean_defect.index, mean_defect.values,
                      color=["#4C72B0","#55A868","#C44E52","#8172B2"])
axes[0, 1].axhline(mean_defect.mean(), color="red", linestyle="--", label="Overall mean")
axes[0, 1].set_title("Average Defect Rate by Machine")
axes[0, 1].set_ylabel("Defect Rate (%)")
axes[0, 1].legend()

# 3. Downtime by machine
total_downtime = df.groupby("machine_id")["downtime_min"].sum()
axes[0, 2].pie(total_downtime.values, labels=total_downtime.index,
               autopct="%1.1f%%", colors=["#4C72B0","#55A868","#C44E52","#8172B2"])
axes[0, 2].set_title("Total Downtime Share by Machine")

# 4. Hourly production heatmap
pivot = df.pivot_table(values="production_units", index="day_of_week",
                       columns="hour", aggfunc="mean")
pivot = pivot.reindex(["Monday", "Tuesday", "Wednesday", "Thursday",
                       "Friday", "Saturday", "Sunday"])
sns.heatmap(pivot, ax=axes[1, 0], cmap="YlOrRd", cbar_kws={"label": "Units"})
axes[1, 0].set_title("Avg Production by Day & Hour")
axes[1, 0].set_xlabel("Hour of Day")

# 5. Temperature vs vibration scatter
scatter_data = df.sample(300)
sc = axes[1, 1].scatter(scatter_data["temperature_c"], scatter_data["vibration_mms"],
                         c=scatter_data["defect_rate"], cmap="Reds", alpha=0.6, s=20)
plt.colorbar(sc, ax=axes[1, 1], label="Defect Rate (%)")
axes[1, 1].set_title("Temperature vs Vibration\n(colored by Defect Rate)")
axes[1, 1].set_xlabel("Temperature (°C)")
axes[1, 1].set_ylabel("Vibration (mm/s)")

# 6. Shift comparison
shift_prod = df.groupby(["shift", "machine_id"])["production_units"].mean().unstack()
shift_prod.plot(kind="bar", ax=axes[1, 2], rot=0)
axes[1, 2].set_title("Avg Production: Day vs Night Shift")
axes[1, 2].set_ylabel("Avg Units/Hour")
axes[1, 2].legend(title="Machine", loc="upper right")

plt.tight_layout()
plt.savefig("day2_quality_dashboard.png", dpi=100, bbox_inches="tight")
plt.show()
print("\n✅ Dashboard saved as day2_quality_dashboard.png")

---
## ✅ Day 2 Exercises

1. **Descriptive Statistics**: Calculate the coefficient of variation (CV = std/mean × 100%) for `vibration_mms` per machine. Which machine is most consistent?
2. **Normal Distribution**: Using the normal distribution model of temperature, find the temperature value at the 99th percentile.
3. **Binomial Distribution**: If the defect probability is 2%, what is the probability of getting **at most 3** defects in a batch of 150 units?
4. **Matplotlib**: Create a multi-line plot showing the daily average `defect_rate` for each machine over the month.
5. **Seaborn**: Create a `catplot` or `FacetGrid` showing the distribution of `production_units` by `shift` for each machine.